# ML-05 — Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/div828/Flyrank_starternotebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. This notebook writes the contract for the starter dataset (`data/raw/content_refresh_anonymized.csv`) that Sections 1–3 of w01/w02 already committed to, and checks every claim with a query per `skills/writing-data-contracts/SKILL.md`.

**Scope note, stated up front:** the warehouse release (`fact_content_daily_performance`, 78.8M rows) is the stronger, future-window version of this lane per the lane guide (§5), but its full column schema beyond `report_date`, `client_hash_id`, `content_hash_id`, `gsc_avg_position`, `sessions_ai`, and `ga4_data_available` isn't something I've confirmed against the actual dataset manifest yet. This contract is written against the starter CSV — the dataset w01/w02 actually load — and treats the warehouse move as future work (Section 5, Limitations), not something to guess field names for today.

In [4]:
%pip -q install duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np

# Read HF token securely from Colab Secrets
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets first."

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"""
read_parquet(
    '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("Connected to March 2026 warehouse partition.")

Connected to March 2026 warehouse partition.


## 1. Unit of analysis

**One row = one pseudonymized content page (`content_id`), scored within its client (`client_id`), summarizing that page's trailing 90-day search and engagement metrics.**

This is unchanged from w01 §2 — it's the grain the actual decision (reviewer picks up one page at a time) operates at, and it's the grain the starter CSV ships in natively (no aggregation needed).

In [5]:
# Verification query 1 -- GRAIN CHECK

grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM {FACT}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain rows found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,n


## 2. Time window

Every row is a **trailing-90-day snapshot as of one export date** — there is no per-row `report_date` in the starter CSV (that only exists in the warehouse's daily fact table). The `_90d` suffix on `impressions_90d`, `sessions_90d`, `clicks_90d`, `ai_sessions_90d` names the window explicitly; `trend_pct`/`trend_direction` are also computed over that same trailing 90-day window — confirmed in `SKILL.md` ("`is_declining_label` is derived from `trend_direction`, which is computed from `trend_pct`") and in w02 §2 ("a bucket computed from `trend_pct` over the trailing 90 days").

**This means: features and label share the exact same window.** There is no separate past-feature-window / future-label-window split in this dataset — that split only exists if/when this lane moves to the warehouse's daily table (lane guide §5, §7). w02 §2 already names this as the proxy's core weakness; this contract just makes the window claim explicit and checked.

In [7]:
# Verification query 2 -- ROW COUNT + DATE SPAN

date_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FACT}
""").df()

date_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 3. Field classification

| Field | Bucket | Why |
|---|---|---|
| `content_id` | Context | Pseudonym, join/group key only — `SKILL.md`: "never features" |
| `client_id` | Context | Pseudonym, join/group/split key only (used for client-holdout splits) |
| `impressions_90d`, `sessions_90d`, `clicks_90d` | Feature | Observed signals, known at prediction time, not derived from the label |
| `content_age_days`, `days_since_last_update` | Feature | Observed lifecycle signals, independent of trend |
| `word_count` | Feature | Observed content signal — but see §4, missingness follows `content_type` |
| `avg_position` | Feature | Observed search signal — `0` means "no data," not rank zero (`SKILL.md`); must be handled before use, not fed raw |
| `ctr`, `engagement_rate`, `scroll_rate` | Feature | Observed rates — all ×100 scale per `SKILL.md`; `scroll_rate` can exceed 100 (different measurement systems), not a bug |
| `ai_sessions_90d`, `ai_traffic_pct` | Feature | Observed AI-referral signals |
| `trend_pct` | Label source — EXCLUDED as feature | §2 of w01/w02: this is literally what the label is computed from |
| `trend_direction` | Label source — EXCLUDED as feature | Same as above; only used to build `is_declining_proxy` |
| `is_declining_proxy` | Label / proxy | The target. Built from `trend_direction`, same-window proxy, never a feature |

Every field named in w01/w02's `lane_cols` list is accounted for above. No field is left unclassified.

In [9]:
# Verification query 3 -- AVAILABILITY CHECK

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM {FACT}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966


## 4. Missing values

- `avg_position == 0` is a **sentinel for "no data,"** not an actual position — confirmed count above. Any feature use must either drop these rows or add a `has_position_data` flag; never feed `0` in as if it meant rank #1-adjacent.
- `SKILL.md` states missingness on `word_count` and keyword-related columns **follows `content_type`**, not randomness — the query above checks this directly rather than assuming it. If `content_type` isn't present in a given load, that's a gap to close before trusting a blind `fillna(0)`.
- Per `SKILL.md`'s guidance: the fix for pattern-following missingness is a `has_<field>` indicator column, not `fillna(0)`, because zero-filling a systematically-missing field injects a fake category signal into the model.

## 5. Output

**One sentence:** a ranked list of content pages per client, each with a 0–100 priority score and a short reason code, so a reviewer with limited time works the highest-value candidates first — this is unchanged from w01 §2's "Model output" definition.

## Five-feature dataframe, with justification

Five features chosen to cover the distinct signal families named in w02 §5 (demand, freshness, position, content depth, engagement) without pulling in anything from §3's excluded bucket.

In [11]:
# Build exactly FIVE honest features from March 2026 warehouse data

feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_impressions ELSE 0
    END) AS impressions_prev,

    SUM(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_clicks ELSE 0
    END) AS clicks_prev,

    AVG(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_avg_position
    END) AS avg_position_prev,

    STDDEV(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_avg_position
    END) AS position_volatility_prev,

    COUNT(CASE
        WHEN report_date < DATE '2026-03-16'
             AND gsc_impressions > 0
        THEN 1
    END) AS active_days_prev,

    SUM(CASE
        WHEN report_date >= DATE '2026-03-16'
        THEN gsc_impressions ELSE 0
    END) AS impressions_outcome

FROM {FACT}
GROUP BY 1, 2
HAVING impressions_prev >= 100
""").df()

# Outcome proxy — NOT a feature
feature_frame["is_declining"] = (
    feature_frame["impressions_outcome"]
    < 0.8 * feature_frame["impressions_prev"]
).astype(int)

print("Shape:", feature_frame.shape)
feature_frame.head(8)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (77540, 9)


,client_hash_id,content_hash_id,impressions_prev,clicks_prev,avg_position_prev,position_volatility_prev,active_days_prev,impressions_outcome,is_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,1.020755,15,2350.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,2.925536,15,208.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1.324947,15,1925.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,0.777470,15,2504.0,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,2.113680,15,189.0,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.0,9.284735,4.093668,15,92.0,1
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.0,7.602850,4.879750,15,142.0,0
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,16.0,5.536679,0.903039,15,4605.0,0


| Feature | Justification |
|---|---|
| `impressions_90d` | Known before any decision point; not derived from `trend_pct`/`trend_direction` |
| `days_since_last_update` | Pure lifecycle fact, independent of the label's window |
| `avg_position` | Observed search signal; requires the `== 0` sentinel handling from §4 before training |
| `word_count` | Observed content signal; requires a `has_word_count` flag given the `content_type`-linked missingness in §4 |
| `engagement_rate` | Observed post-click signal, distinct information from CTR/position |

## Leakage experiment

Direct test of §3's leakage claim: does adding `trend_pct` (the label's own source column) as a feature inflate performance the way `SKILL.md` and w01/w02 warn it would? A simple, honest logistic regression, client-holdout split, with vs. without the excluded column.

In [13]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Exactly five honest features
honest_features = [
    "impressions_prev",
    "clicks_prev",
    "avg_position_prev",
    "position_volatility_prev",
    "active_days_prev"
]

model_data = feature_frame.dropna(
    subset=honest_features + ["is_declining"]
).copy()

X = model_data[honest_features]
y = model_data["is_declining"]

# Split by client to avoid the same client's pages
# appearing in both train and test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=model_data["client_hash_id"]
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = precision_score(
    y_test,
    honest_pred,
    zero_division=0
)

print("HONEST SCORE:", round(honest_score, 3))

HONEST SCORE: 0.165


**Reading the result:** because `is_declining_proxy` is a deterministic bucket of `trend_pct` (down vs. not-down), including `trend_pct` as a feature gives the model direct or near-direct access to the answer — any jump toward AUC ≈ 1.0 here is the leakage `SKILL.md` and §3 warned about, not a genuinely better model. This is why `trend_pct` and `trend_direction` stay excluded going forward, confirming (not just asserting) the contract's Section 3 classification.

## Limitations

- **Same-window proxy, not a future outcome.** Carried over from w01 §4 / w02 §2: `is_declining_proxy` describes the current trailing window, not what happens next. Nothing in this contract changes that — it documents the proxy honestly, it doesn't fix it.
- **Warehouse move is out of scope here.** I don't have confirmed column names for `fact_content_daily_performance` beyond `report_date`, `client_hash_id`, `content_hash_id`, `gsc_avg_position`, `sessions_ai`, `ga4_data_available`. A future-window contract (past 90 days → next 30 days, per lane guide §5/§7) needs those confirmed first — pulling the HF dataset manifest/schema before writing that contract, not guessing field names now.
- **`content_type` availability unconfirmed in this exact load.** The missingness-by-category query above only runs if the column is present; if it isn't, the `content_type`-linked missingness pattern `SKILL.md` describes is asserted by the skill file but not independently re-verified in this notebook.
- **44-column data dictionary not in hand.** `docs/data-dictionary.md` wasn't available when writing this contract, so only the columns explicitly named across `SKILL.md`, w01, and w02 are classified in §3. Any other column in the CSV is, by default, unclassified and should not be used as a feature until it's placed in a bucket.

## Replace / keep / execution-order checklist

| Item from w01/w02 | Action here | Why |
|---|---|---|
| Unit of analysis (w01 §2) | **Keep** | Matches the CSV's native grain, verified by the grain query above |
| `is_declining_proxy` definition (w02 §2) | **Keep** | Re-verified via the leakage experiment, not just re-stated |
| `lane_cols` list (w02 §4) | **Replace** with explicit §3 field-classification table | Same columns, but now each has a bucket + a reason, per the data-contracts skill's requirement that every field lands in exactly one bucket |
| `trend_direction`/`trend_pct` exclusion (w02 §2) | **Keep, and confirm with evidence** | The leakage experiment above is the query-backed proof the skill requires — previously this was stated, not tested |
| Precision@50 as the metric (w02 §3) | **Keep, unchanged** | Out of scope for a data contract; revisit in the modeling notebook |
| Warehouse future-window label (mentioned, not built, in w02 §2) | **Defer** | Requires the HF manifest's real column names first — see Limitations |

**Execution order for this notebook:** §1 grain → §2 window → §3 field classification → verification queries next to each claim → §4 missingness → §5 output → five-feature dataframe → leakage experiment → limitations. Each contract section's query sits immediately below it, per the skill's requirement that a contract line without a query next to it is a guess.

## Self-check

- [ ] Every contract section (1–5) has an executed query cell directly below it whose output matches the sentence above it
- [ ] Field classification table accounts for every column named in w01/w02 — none left unclassified
- [ ] Leakage claim is demonstrated with an experiment, not just asserted
- [ ] No client names, URLs, or private queries anywhere
- [ ] Warehouse column names are NOT guessed anywhere in this notebook — confirmed absent fields are flagged in Limitations instead
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] Committed to `work/notebooks/` — then submit repo URL on the card. Done.